In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from catboost import CatBoostClassifier, Pool

df = pd.read_json('../data/All_Beauty.jsonl', lines=True)

df = df.dropna(subset=['rating'])
df['rating'] = df['rating'].astype(int)

df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')
df['full_text'] = df['title'] + " " + df['text']

df['text_length'] = df['full_text'].apply(lambda x: len(x.split()))


X = df[['full_text', 'text_length']]
y = df['rating']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y 
)

text_cols = ['full_text']

train_pool = Pool(data=X_train, label=y_train, text_features=text_cols)
test_pool = Pool(data=X_test, label=y_test, text_features=text_cols)

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    eval_metric='Accuracy',
    auto_class_weights='Balanced',
    task_type='GPU',
    early_stopping_rounds=50,
    verbose=100
)

model.fit(train_pool, eval_set=test_pool)

predictions = model.predict(X_test)

feature_importances = model.get_feature_importance(train_pool)
for score, name in sorted(zip(feature_importances, X.columns), reverse=True):
    print(f"{name}: {score:.2f}%")

0:	learn: 0.5621429	test: 0.5743092	best: 0.5743092 (0)	total: 376ms	remaining: 6m 15s
100:	learn: 0.5905196	test: 0.6027112	best: 0.6028758 (98)	total: 7.65s	remaining: 1m 8s
200:	learn: 0.5996340	test: 0.6085565	best: 0.6085935 (194)	total: 13.9s	remaining: 55.4s
300:	learn: 0.6071365	test: 0.6115200	best: 0.6117406 (295)	total: 20.3s	remaining: 47s
400:	learn: 0.6131675	test: 0.6147784	best: 0.6147784 (400)	total: 26.4s	remaining: 39.4s
500:	learn: 0.6186229	test: 0.6160972	best: 0.6161525 (483)	total: 32.5s	remaining: 32.4s
bestTest = 0.6167407039
bestIteration = 517
Shrink model to first 518 iterations.
full_text: 94.15%
text_length: 5.85%


In [4]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           1       0.78      0.73      0.75     20416
           2       0.34      0.49      0.40      8607
           3       0.43      0.50      0.47     11261
           4       0.38      0.55      0.45     15876
           5       0.94      0.81      0.87     84146

    accuracy                           0.72    140306
   macro avg       0.57      0.62      0.59    140306
weighted avg       0.77      0.72      0.74    140306

